# Setup

In [1]:
%%capture
!pip install google-api-python-client>=2.0
!pip install google-auth>=2.0
!pip install google-auth-httplib2>=0.2
!pip install google-auth-oauthlib>=1.0
!pip install PytorchWildlife

In [2]:
from pathlib import Path
from typing import Any, Iterator
import json
import shutil
import sys
import time
import torch
from tqdm.auto import tqdm
from PytorchWildlife.models import detection as pw_detection

# from google_drive_client import GoogleDriveClient, DriveClientInitArgs
from google.colab import drive
drive.mount('/content/drive')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [3]:
ORIGINAL_MAP_PATH = Path("data/dirs.json")
PREPROCESSED_MAP_PATH = Path("data/dirs_preprocessed.json")
OUTPUT_PATH = Path("data/wild_detector_results.json")
TEMP_PATH = Path(".tmp/wild_detector")

DOWNLOAD_BATCH_SIZE = 16
DOWNLOAD_WORKERS = 6
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

TEMP_PATH.mkdir(parents=True, exist_ok=True)


# Helpers


In [4]:
def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r") as file:
        return json.load(file)


def save_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )
    temporary_path.replace(path)


def chunks(values: list, size: int) -> Iterator[list]:
    for index in range(0, len(values), size):
        yield values[index:index + size]


def to_serializable(value: Any) -> list:
    if hasattr(value, "detach"):
        value = value.detach().cpu()
    if hasattr(value, "tolist"):
        return value.tolist()
    return list(value)

In [5]:
def extract_keys_path(name, map, keys):
    keys.append(name)
    if isinstance(map[name], list):
        return keys
    for key in map[name]:
        result = extract_keys_path(key, map[name], keys)
        if result:
            return result
    keys.pop()
    return None


def extract_last_value(map: dict, keys: list[str]):
    curr_map = map
    for key in keys:
        curr_map = curr_map[key]
    return [val for val in curr_map if not val.startswith("._")]

def check_routes(fpath: str, name: str) -> dict:
    path = Path(fpath) / name

    entries = [item for item in path.iterdir()]

    if any(not f.is_dir() for f in entries):
        return {name: [f.name for f in entries if not f.is_dir()]}

    dirs = [f.name for f in entries if f.is_dir()]
    children = {}
    for cname in dirs:
        children.update(check_routes(path, cname))
    return {name: children}



# Load directories information

In [8]:
DOWNLOAD_PATH_MAP = False
PREPROCESS_PATH_MAP = True
BASE_PATH = Path("drive/MyDrive/ECHO")
SOURCE = BASE_PATH / "Data"
START_DIR = "Sin clasificar"


if DOWNLOAD_PATH_MAP:
    map = check_routes(SOURCE, START_DIR)
    result = {}
    result[str(SOURCE)] = map
    save_json(ORIGINAL_MAP_PATH, result)
else:
    map = load_json(ORIGINAL_MAP_PATH)[str(SOURCE)][START_DIR]
if PREPROCESS_PATH_MAP:
    preprocessed_map = {}
    for key in map.keys():
        keys = extract_keys_path(key, map, [])
        values = extract_last_value(map, keys)  # type: ignore

        dirpath = f"{SOURCE}/{START_DIR}/" + "/".join(keys)  # type: ignore
        preprocessed_map[dirpath] = values
    save_json(PREPROCESSED_MAP_PATH, preprocessed_map)
else:
    preprocessed_map = load_json(PREPROCESSED_MAP_PATH)

# Model detector

In [10]:
# Load the detector model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
detector = pw_detection.MegaDetectorV6(device=DEVICE, version="MDV6-yolov10-e")
results: dict[str, list[dict]] = load_json(OUTPUT_PATH)

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10x summary (fused): 191 layers, 29,399,417 parameters, 0 gradients, 160.0 GFLOPs


# Detection

In [11]:
LABEL_THRESHOLDS = {
    "animal": 0.25,
    "person": 0.25,
    "vehicle": 0.25,
}
MEGADETECTOR_LABELS = {
    0: "animal",
    1: "person",
    2: "vehicle",
}

DEFAULT_THRESHOLD = 0.25

In [12]:
def safe_to_list(value):
    if value is None:
        return []
    if hasattr(value, "detach"):
        value = value.detach().cpu()
    if hasattr(value, "tolist"):
        return value.tolist()
    return list(value)


def extract_detection_metadata(detections) -> dict:
    boxes = safe_to_list(detections.xyxy)
    confidences = safe_to_list(detections.confidence)
    class_ids = safe_to_list(detections.class_id)

    detection_items = []
    max_confidence_by_label = {
        label: 0.0
        for label in LABEL_THRESHOLDS
    }

    for box, confidence, class_id in zip(boxes, confidences, class_ids):
        class_id = int(class_id)
        confidence = float(confidence)

        label = MEGADETECTOR_LABELS.get(
            class_id,
            f"unknown_{class_id}"
        )

        threshold = LABEL_THRESHOLDS.get(
            label,
            DEFAULT_THRESHOLD
        )

        passed_threshold = confidence >= threshold

        item = {
            "label": label,
            "class_id": class_id,
            "confidence": confidence,
            "bbox": [float(x) for x in box],
            "passed_threshold": passed_threshold,
        }

        detection_items.append(item)

        if passed_threshold and label in max_confidence_by_label:
            max_confidence_by_label[label] = max(
                max_confidence_by_label[label],
                confidence,
            )

    labels = sorted([
        label
        for label, max_confidence in max_confidence_by_label.items()
        if max_confidence >= LABEL_THRESHOLDS[label]
    ])

    return {
        "labels": labels,
        "all_detections": detection_items,
        "max_confidence_by_label": max_confidence_by_label,
    }

def detect_image(image_path: Path, folder_path: str | None = None) -> dict:
    with torch.inference_mode():
        result = detector.single_image_detection(str(image_path))

    detections = result["detections"]
    print(detections)

    detection_metadata = extract_detection_metadata(detections)

    return {
        "name": image_path.name,
        "folder_path": folder_path,
        "source_path": (
            f"{folder_path}/{image_path.name}"
            if folder_path is not None
            else str(image_path)
        ),
        "labels": detection_metadata["labels"],
        "has_detection": len(detection_metadata["labels"]) > 0,
        "has_animal": "animal" in detection_metadata["labels"],
        "has_person": "person" in detection_metadata["labels"],
        "has_vehicle": "vehicle" in detection_metadata["labels"],
        "all_detections": detection_metadata["all_detections"],
        "max_confidence_by_label": detection_metadata["max_confidence_by_label"],
    }

In [14]:
for folder_path, expected_names in preprocessed_map.items():
    print(f"\nProcessing: {folder_path}")
    folder_path = Path(folder_path)
    folder_key = str(folder_path)
    folder_results = results.setdefault(folder_key, [])

    already_processed = {
        item["name"]
        for item in folder_results
        if "name" in item
    }

    expected_names = {
        name
        for name in expected_names
        if not name.startswith("._")
        and Path(name).suffix.lower() in VALID_EXTENSIONS
        and name not in already_processed
    }

    drive_entries = {
        entry.name: entry
        for entry in folder_path.iterdir() if entry.is_file()
    }

    pending_entries = [
        drive_entries[name]
        for name in sorted(expected_names)
        if name in drive_entries
    ]

    missing_names = sorted(expected_names.difference(drive_entries))

    print("Pending images:", len(pending_entries))
    print("Missing images:", len(missing_names))

    for batch_index, batch in enumerate(
        chunks(pending_entries, DOWNLOAD_BATCH_SIZE)
    ):
        batch_path = TEMP_PATH / f"batch_{batch_index:05d}"
        batch_path.mkdir(parents=True, exist_ok=True)

        try:
            for entry in tqdm(batch, desc=Path(folder_path).name, leave=False):
                local_path = folder_path / entry.name

                if local_path is None:
                    print(f"Download failed: {entry.name}")
                    continue

                try:
                    start_time = time.perf_counter()
                    meta = detect_image(local_path)
                    folder_results.append(meta)

                except Exception as error:
                    print(
                        f"Detection error in {entry.name}: "
                        f"{error}"
                    )

            save_json(OUTPUT_PATH, results)

        finally:
            shutil.rmtree(
                batch_path,
                ignore_errors=True,
            )

    print(
        f"Stored detections: "
        f"{len(folder_results)}"
    )

save_json(OUTPUT_PATH, results)

print("\nFinished.")
print("Results:", OUTPUT_PATH)


Processing: drive/MyDrive/ECHO/Data/Sin clasificar/CAM01_02
Pending images: 1
Missing images: 0


CAM01_02:   0%|          | 0/1 [00:00<?, ?it/s]

Detection error in 03040448.JPG: cannot identify image file 'drive/MyDrive/ECHO/Data/Sin clasificar/CAM01_02/03040448.JPG'
Stored detections: 369

Processing: drive/MyDrive/ECHO/Data/Sin clasificar/CAM03/DCIM/100EK113
Pending images: 940
Missing images: 0


100EK113:   0%|          | 0/16 [00:00<?, ?it/s]


0: 1280x1280 2 animals, 266.5ms
Speed: 11.7ms preprocess, 266.5ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([[     3213.8,      2279.8,      3341.8,      2437.3],
       [     1648.3,      2844.9,      1794.8,      3240.4]], dtype=float32), mask=None, confidence=array([    0.38734,     0.20394], dtype=float32), class_id=array([0, 0]), tracker_id=None, data={})

0: 1280x1280 (no detections), 228.8ms
Speed: 10.5ms preprocess, 228.8ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=None, confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=None, data={})

0: 1280x1280 (no detections), 230.2ms
Speed: 10.8ms preprocess, 230.2ms inference, 0.7ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=None, confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_i

...


0: 1280x1280 (no detections), 240.7ms
Speed: 14.8ms preprocess, 240.7ms inference, 0.9ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=None, confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=None, data={})

0: 1280x1280 (no detections), 235.1ms
Speed: 16.2ms preprocess, 235.1ms inference, 0.7ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=None, confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=None, data={})

0: 1280x1280 (no detections), 236.2ms
Speed: 15.8ms preprocess, 236.2ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=None, confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=None, data={})

0: 1280x1280 (no detections), 235.5ms
Speed: 15.8ms preprocess, 235.5ms inference, 1.0ms p

100EK113:   0%|          | 0/16 [00:00<?, ?it/s]


0: 1280x1280 1 animal, 239.1ms
Speed: 15.5ms preprocess, 239.1ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([[     51.785,      2.7395,      2751.2,      2046.3]], dtype=float32), mask=None, confidence=array([    0.67813], dtype=float32), class_id=array([0]), tracker_id=None, data={})

0: 1280x1280 1 person, 236.1ms
Speed: 13.5ms preprocess, 236.1ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([[     673.03,      3.1545,      6315.3,      4029.9]], dtype=float32), mask=None, confidence=array([    0.82196], dtype=float32), class_id=array([1]), tracker_id=None, data={})

0: 1280x1280 1 person, 236.9ms
Speed: 13.9ms preprocess, 236.9ms inference, 1.1ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([[     670.16,      2.6871,        6318,      4028.8]], dtype=float32), mask=None, confidence=array([     0.8414], dtype=float32), class_id=array([1]), tracker_id=None, data={})



100EK113:   0%|          | 0/10 [00:00<?, ?it/s]


0: 1280x1280 1 person, 236.5ms
Speed: 24.7ms preprocess, 236.5ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([[     148.06,           0,      3020.2,      2048.9]], dtype=float32), mask=None, confidence=array([    0.77292], dtype=float32), class_id=array([1]), tracker_id=None, data={})

0: 1280x1280 1 animal, 1 person, 235.1ms
Speed: 22.1ms preprocess, 235.1ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([[     418.73,      675.57,        2771,      2045.4],
       [     418.73,      675.57,        2771,      2045.4]], dtype=float32), mask=None, confidence=array([    0.45364,     0.32538], dtype=float32), class_id=array([1, 0]), tracker_id=None, data={})

0: 1280x1280 (no detections), 239.1ms
Speed: 20.5ms preprocess, 239.1ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=None, confidence=array([], dtype=float32), cla

In [17]:
shutil.copy(
    Path('data/wild_detector_results.json'),
    Path('drive/MyDrive/ECHO/Metadata/wild_detections/wild_detector_results.json')
    )

PosixPath('drive/MyDrive/ECHO/Metadata/wild_detections/wild_detector_results.json')

# Load results and filter

In [18]:
results = load_json(Path('data/wild_detector_results.json'))

In [19]:
for k,v in results.items():
    print(k, len(v))

drive/MyDrive/ECHO/Data/Sin clasificar/CAM01_02 369
drive/MyDrive/ECHO/Data/Sin clasificar/CAM03/DCIM/100EK113 999
drive/MyDrive/ECHO/Data/Sin clasificar/CAM04/100EK113 999
drive/MyDrive/ECHO/Data/Sin clasificar/CAM05/100EK113 275
drive/MyDrive/ECHO/Data/Sin clasificar/CAM07/DCIM/100EK113 956
drive/MyDrive/ECHO/Data/Sin clasificar/CAM08/100EK113 120
drive/MyDrive/ECHO/Data/Sin clasificar/CAM09/DCIM/100EK113 495
drive/MyDrive/ECHO/Data/Sin clasificar/CAM10/DCIM/100EK113 151
drive/MyDrive/ECHO/Data/Sin clasificar/CAM11/DCIM/100EK113 244
drive/MyDrive/ECHO/Data/Sin clasificar/CAM12/DCIM/100EK113 250
drive/MyDrive/ECHO/Data/Sin clasificar/CAM13/DCIM/100EK113 457
drive/MyDrive/ECHO/Data/Sin clasificar/CAM14/DCIM/100EK113 811
drive/MyDrive/ECHO/Data/Sin clasificar/CAM15/DCIM/100EK113 360
drive/MyDrive/ECHO/Data/Sin clasificar/CAM16/DCIM/100EK113 106


In [59]:
DETECTIONS = Path('/content/drive/MyDrive/ECHO/Metadata/wild_detections/wild_detector_results.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/ECHO/Data/Sin clasificar filtered')

# ------ Important: Threshold parameters! ------
THRESHOLD_LABELS = {
    'animal': 0.65,
    'person': 0.60,
}
# ------------------------------------------------

In [60]:

detections = load_json(DETECTIONS)

filtered_detections = {}

for path, images_meta in detections.items():
    filtered_detections[path] = []

    for image_meta in images_meta:
        has_detection = image_meta['has_detection']
        has_animal = image_meta['has_animal']
        has_person = image_meta['has_person']

        max_conf_by_label = image_meta['max_confidence_by_label']
        threshold_cnstr = (
            (
                max_conf_by_label['animal'] >= THRESHOLD_LABELS['animal'] and
                max_conf_by_label['person'] >= THRESHOLD_LABELS['person'])
            or
            (
                max_conf_by_label['animal'] >= THRESHOLD_LABELS['animal'] and
                max_conf_by_label['person'] < THRESHOLD_LABELS['person']
            )
        )

        if not threshold_cnstr:
            continue

        if not has_detection:
            continue

        if has_person and not has_animal:
            continue

        valid = (
            (has_animal and not has_person) or
            (has_animal and has_person)
        )

        if not valid:
            continue

        filtered_detections[path].append(image_meta)



count_imgs_before = 0
for k, v in detections.items():
    count_imgs_before += len(v)

count_imgs_after = 0
for k, v in filtered_detections.items():
    count_imgs_after += len(v)

print(f'Number of images before filtering: {count_imgs_before}')
print(f'Number of images after filtering: {count_imgs_after}')


Number of images before filtering: 6592
Number of images after filtering: 1009


In [62]:
import os
if os.path.exists(OUTPUT_DIR):
    print(f'Directory {OUTPUT_DIR} already exists')
else:
    os.mkdir(OUTPUT_DIR)
    print(f'Directory {OUTPUT_DIR} created')


Directory /content/drive/MyDrive/ECHO/Data/Sin clasificar filtered created


In [64]:
for current_dir, images in tqdm(filtered_detections.items(), desc="Processing directories"):
    base_name = current_dir.replace('/', '_')
    print(f"Starting copy from {current_dir}")
    for img in images:
        img_name = img['name']
        source_path = os.path.join(current_dir, img_name)
        file_name = base_name + '_' + img_name
        # we validate if the image already exists
        dest_path = os.path.join(OUTPUT_DIR, file_name)
        if os.path.exists(dest_path):
            #print(f'Image {file_name} already exists')
            continue
        shutil.copy(source_path, dest_path)
        #print(f'Image {file_name} was copied successfully')
    print(f"Copy process has ended successfully at {current_dir}")


Processing directories:   0%|          | 0/14 [00:00<?, ?it/s]

Starting copy from drive/MyDrive/ECHO/Data/Sin clasificar/CAM01_02
Copy process has ended successfully at drive/MyDrive/ECHO/Data/Sin clasificar/CAM01_02
Starting copy from drive/MyDrive/ECHO/Data/Sin clasificar/CAM03/DCIM/100EK113
Copy process has ended successfully at drive/MyDrive/ECHO/Data/Sin clasificar/CAM03/DCIM/100EK113
Starting copy from drive/MyDrive/ECHO/Data/Sin clasificar/CAM04/100EK113
Copy process has ended successfully at drive/MyDrive/ECHO/Data/Sin clasificar/CAM04/100EK113
Starting copy from drive/MyDrive/ECHO/Data/Sin clasificar/CAM05/100EK113
Copy process has ended successfully at drive/MyDrive/ECHO/Data/Sin clasificar/CAM05/100EK113
Starting copy from drive/MyDrive/ECHO/Data/Sin clasificar/CAM07/DCIM/100EK113
Copy process has ended successfully at drive/MyDrive/ECHO/Data/Sin clasificar/CAM07/DCIM/100EK113
Starting copy from drive/MyDrive/ECHO/Data/Sin clasificar/CAM08/100EK113
Copy process has ended successfully at drive/MyDrive/ECHO/Data/Sin clasificar/CAM08/100EK